# Lesson 7 — Pipeline Orchestration

**Learning objective:** Understand how `PipelineConfig` and `run_pipeline` wire all seven phases into a single callable. Learn which phase toggles matter for different use cases, and see how the pipeline handles partial runs, failures, and phase skips gracefully.

---

## The full pipeline

```
PipelineConfig
    │
    ├─ [Phase 0]  build_sector_ontology()   optional, expensive — Exa + extra Bedrock calls
    ├─ [Phase 2]  embed_ptcs()              Bedrock Titan, resume-safe
    ├─ [Phases 3-5]  cluster_ptcs()        HDBSCAN, cross-map, sector-tag
    │                 → canonical_df        the canonical mechanism table
    ├─ [Phase 6]  build_timeseries()        mention matrix + earnings alignment
    ├─ [Phase 7A] run_granger_analysis()    quarterly Granger with 3 fixes
    └─ [Phase 7B] run_event_study()         daily CAR, t-test per canonical
    │
    └─▶ PipelineResult
            canonical_df, mention_matrix, aligned_df,
            granger_df, eventstudy_df, sector_vectors
```

Every phase is independently togglable. The pipeline is data-layer only — it writes CSVs and NPY files. Analysis, plotting, and reporting live in notebooks or a Streamlit app.

In [ ]:
import sys, json, tempfile
from pathlib import Path
from datetime import date
from dataclasses import fields
from unittest.mock import patch, MagicMock

import numpy as np
import pandas as pd

sys.path.insert(0, '../../src')

from signal_intelligence.pipeline import PipelineConfig, PipelineResult, run_pipeline

---

## Exercise 1 — Inspect PipelineConfig

`PipelineConfig` is a dataclass. Every field has a sensible default — the minimum required to construct one is just `ticker` and `pool_path`.

In [ ]:
# Show all configurable fields
print("PipelineConfig fields:")
print(f"{'Field':<30}  {'Default':<20}  {'Type'}")
print("-" * 70)
for f in fields(PipelineConfig):
    if f.name == "sector_vectors":
        default_str = "None"
    elif hasattr(f.default, '__name__'):
        default_str = f.default.__name__ + "()"
    elif f.default is not f.default_factory if hasattr(f, 'default_factory') else True:
        try:
            default_str = str(f.default) if f.default is not f.default_factory else "<factory>"
        except Exception:
            default_str = "<factory>"
    else:
        default_str = "<required>"
    type_str = str(f.type) if isinstance(f.type, str) else f.type.__name__ if hasattr(f.type, '__name__') else str(f.type)
    print(f"  {f.name:<28}  {default_str:<20}  {type_str}")

In [ ]:
# Minimal config: only required fields
minimal_cfg = PipelineConfig(
    ticker="ML.PA",
    pool_path=Path("data/processed/ptc_pool.jsonl"),
)

print("Minimal config:")
print(f"  ticker              = {minimal_cfg.ticker}")
print(f"  run_embedding       = {minimal_cfg.run_embedding}")
print(f"  run_clustering      = {minimal_cfg.run_clustering}")
print(f"  run_sector_ontology = {minimal_cfg.run_sector_ontology}")
print(f"  min_cluster_size    = {minimal_cfg.min_cluster_size}")
print(f"  cross_map_threshold = {minimal_cfg.cross_map_threshold}")
print(f"  region              = {minimal_cfg.region}")

# Full analysis config for a complete run
full_cfg = PipelineConfig(
    ticker="ML.PA",
    company_name="Michelin",
    sector="Specialty Chemicals & Tires",
    pool_path=Path("data/processed/ptc_pool.jsonl"),
    run_sector_ontology=True,   # Phase 0: peer discovery + sector vectors
    run_embedding=True,
    run_clustering=True,
    run_timeseries=True,
    run_granger=True,
    run_eventstudy=True,
    min_cluster_size=3,
    cross_map_threshold=0.82,
    granger_alpha=0.05,
    sparsity_threshold=0.30,
    max_lag=4,
)
print("\nFull config ready.")

---

## Exercise 2 — Phase toggles: embedding-only run

A common use case during development: you've already clustered and just want to re-embed new PTCs without re-running everything downstream.

The pipeline skips all disabled phases and returns a partial `PipelineResult`. It will try to load a previously saved canonical CSV from disk.

In [ ]:
# Build the fake intermediate objects that would exist after prior runs

def make_fake_canonical_df(n=4):
    return pd.DataFrame([
        {
            "canonical_id": f"ML.PA-management-{i}",
            "ticker": "ML.PA",
            "corpus": "management",
            "direction": "precursor" if i % 2 == 0 else "consequence",
            "polarity": -1 if i < 2 else 1,
            "corpus_presence": "triangulated",
            "sector_tag": "untagged",
            "n_ptcs": 3,
            "is_noise": False,
            "representative_mechanism": f"Mechanism {i} text here.",
            "source_ids": json.dumps([f"ML.PA-mgmt-2024-0{i}"]),
            "matched_canonical_id": None,
            "cross_map_similarity": None,
        }
        for i in range(n)
    ])

fake_vectors = np.random.default_rng(0).random((10, 1024), dtype=float).astype(np.float32)

with tempfile.TemporaryDirectory() as tmpdir:
    pool_path = Path(tmpdir) / "pool.jsonl"
    pool_path.write_text("")  # empty placeholder
    interim = Path(tmpdir) / "interim"
    processed = Path(tmpdir) / "processed"
    processed.mkdir(parents=True)

    # Pre-write a canonical CSV as if clustering already ran
    canonical_csv = processed / "ML.PA_canonicals.csv"
    make_fake_canonical_df().to_csv(canonical_csv, index=False)

    cfg = PipelineConfig(
        ticker="ML.PA",
        pool_path=pool_path,
        interim_dir=interim,
        processed_dir=processed,
        run_embedding=True,
        run_clustering=False,   # ← skip clustering, load from disk instead
        run_timeseries=False,
        run_granger=False,
        run_eventstudy=False,
    )

    with patch("signal_intelligence.pipeline.embed_ptcs", return_value=(fake_vectors, pd.DataFrame())) as mock_embed:
        result = run_pipeline(cfg)

print(f"embed_ptcs called  : {mock_embed.called}")
print(f"canonical_df shape : {result.canonical_df.shape}  ← loaded from disk")
print(f"granger_df empty   : {result.granger_df.empty}  ← skipped")
print(f"eventstudy_df empty: {result.eventstudy_df.empty}  ← skipped")

---

## Exercise 3 — End-to-end mock run

Run the full pipeline with all phases enabled, mocking every external call. This demonstrates the complete data flow through a single `run_pipeline()` call.

In [ ]:
from signal_intelligence.eventstudy import _daily_returns
from ptc_extraction.models import PTC, Direction

# Build realistic fake objects for each phase
rng = np.random.default_rng(42)
trading_days = pd.date_range("2022-01-01", "2025-01-01", freq="B")
prices_df = pd.DataFrame(
    {"adj_close": (1.0 + pd.Series(rng.normal(0.0003, 0.012, len(trading_days)), index=trading_days)).cumprod() * 100},
    index=trading_days,
)

quarters = pd.date_range("2018-03-31", periods=22, freq="QE")
earnings_df = pd.DataFrame({
    "date": quarters,
    "surprise_pct": rng.normal(0, 2, len(quarters)),
})

fake_canonical = make_fake_canonical_df(4)
fake_vectors = rng.random((10, 1024), dtype=float).astype(np.float32)

def make_aligned(canonical_df, quarters):
    rows = []
    for _, canon in canonical_df.iterrows():
        for q in quarters:
            rows.append({"canonical_id": canon["canonical_id"], "quarter_end": q,
                         "mention_count": float(rng.integers(0, 4)),
                         "surprise_pct": float(rng.normal(0, 2))})
    return pd.DataFrame(rows)

fake_aligned = make_aligned(fake_canonical, quarters)
fake_mention_matrix = fake_canonical[["canonical_id"]].copy()

event_dates = [pd.Timestamp("2023-03-15"), pd.Timestamp("2023-11-10"),
               pd.Timestamp("2024-02-22"), pd.Timestamp("2024-08-01")]
fake_ptcs = [
    PTC(
        mechanism=f"Mechanism for event {i}.",
        raw_text="raw text",
        direction=Direction.precursor,
        polarity=-1,
        source_id=f"ML.PA-mgmt-2024-0{i}",
        source_date=event_dates[i].date(),
        source_type="management",
        extracted_by="mock",
        extraction_version="v1.1",
    )
    for i in range(4)
]

with tempfile.TemporaryDirectory() as tmpdir:
    pool_path = Path(tmpdir) / "pool.jsonl"
    pool_path.write_text("")
    interim = Path(tmpdir) / "interim"
    processed = Path(tmpdir) / "processed"
    market = Path(tmpdir) / "market"

    cfg = PipelineConfig(
        ticker="ML.PA",
        pool_path=pool_path,
        interim_dir=interim,
        processed_dir=processed,
        market_dir=market,
        run_sector_ontology=False,
        run_embedding=True,
        run_clustering=True,
        run_timeseries=True,
        run_granger=True,
        run_eventstudy=True,
    )

    with (
        patch("signal_intelligence.pipeline.embed_ptcs", return_value=(fake_vectors, pd.DataFrame())),
        patch("signal_intelligence.pipeline.cluster_ptcs", return_value=fake_canonical),
        patch("signal_intelligence.pipeline.get_prices", return_value=prices_df),
        patch("signal_intelligence.pipeline.get_earnings", return_value=earnings_df),
        patch("signal_intelligence.pipeline.build_timeseries", return_value=(fake_mention_matrix, fake_aligned)),
        patch("signal_intelligence.eventstudy.load_pool", return_value=fake_ptcs),
    ):
        result = run_pipeline(cfg)

print("Pipeline complete. PipelineResult contents:")
print(f"  canonical_df   : {result.canonical_df.shape}")
print(f"  mention_matrix : {result.mention_matrix.shape}")
print(f"  aligned_df     : {result.aligned_df.shape}")
print(f"  granger_df     : {result.granger_df.shape}")
print(f"  eventstudy_df  : {result.eventstudy_df.shape}")
print(f"  sector_vectors : {result.sector_vectors.shape}")

---

## Exercise 4 — Phase failure isolation

If market data fetching fails (e.g. no FMP_API_KEY), Phases 6 and 7 are skipped gracefully. The canonical table is still returned. The pipeline never crashes on a single-phase failure.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    pool_path = Path(tmpdir) / "pool.jsonl"
    pool_path.write_text("")

    cfg = PipelineConfig(
        ticker="ML.PA",
        pool_path=pool_path,
        interim_dir=Path(tmpdir) / "interim",
        processed_dir=Path(tmpdir) / "processed",
        market_dir=Path(tmpdir) / "market",
        run_embedding=True,
        run_clustering=True,
        run_timeseries=True,
        run_granger=True,
        run_eventstudy=True,
    )

    with (
        patch("signal_intelligence.pipeline.embed_ptcs", return_value=(fake_vectors, pd.DataFrame())),
        patch("signal_intelligence.pipeline.cluster_ptcs", return_value=fake_canonical),
        patch("signal_intelligence.pipeline.get_prices", side_effect=Exception("No API key")),
        patch("signal_intelligence.pipeline.get_earnings", side_effect=Exception("No API key")),
    ):
        result = run_pipeline(cfg)

print("Pipeline completed despite market data failure:")
print(f"  canonical_df   : {result.canonical_df.shape}  ← populated")
print(f"  aligned_df     : {result.aligned_df.shape}  ← empty (no market data)")
print(f"  granger_df     : {result.granger_df.shape}  ← empty (no aligned_df)")
print(f"  eventstudy_df  : {result.eventstudy_df.shape}  ← empty (no prices)")

---

## Real-world usage pattern

```python
from pathlib import Path
from signal_intelligence.pipeline import PipelineConfig, run_pipeline

# First run: full pipeline (takes ~30min with real Bedrock + market data)
cfg = PipelineConfig(
    ticker="ML.PA",
    company_name="Michelin",
    sector="Specialty Chemicals & Tires",
    pool_path=Path("data/processed/ptc_pool.jsonl"),
)
result = run_pipeline(cfg)

# Re-run with new PTCs added to the pool (resume-safe: skips already-embedded)
result2 = run_pipeline(cfg)   # embed_ptcs skips existing hashes automatically

# Analysis-only run: clustering already done, skip expensive phases
analysis_cfg = PipelineConfig(
    ticker="ML.PA",
    pool_path=Path("data/processed/ptc_pool.jsonl"),
    run_embedding=False,
    run_clustering=False,    # loads ML.PA_canonicals.csv from disk
    run_timeseries=True,
    run_granger=True,
    run_eventstudy=True,
)
analysis_result = run_pipeline(analysis_cfg)
print(analysis_result.eventstudy_df[
    ["canonical_id", "direction", "n_events", "mean_car", "p_value"]
].sort_values("mean_car"))
```

---

## Pipeline output directory layout

```
data/
  interim/
    {ticker}_embeddings.npy          float32 array (N, 1024)
    {ticker}_embedding_index.csv     N rows: content_hash, source_id, direction, ...
    sector_embeddings.npy            Phase 0 (optional)
    sector_embedding_index.csv
  processed/
    {ticker}_canonicals.csv          Phases 3-5 output
    {ticker}_mention_matrix.csv      Phase 6
    {ticker}_aligned.csv             Phase 6
    {ticker}_granger.csv             Phase 7A
    {ticker}_eventstudy.csv          Phase 7B
  market/
    {ticker}_prices.csv              from FMP API
    {ticker}_earnings.csv
```

All files are re-creatable from `data/processed/ptc_pool.jsonl` and market data — the pool is the single source of truth.

---

## Key takeaways

1. `PipelineConfig` is the single configuration object for a complete ticker analysis — change one field to toggle any phase.
2. Phase toggles let you iterate fast: disable expensive phases (embedding, sector ontology) when debugging downstream stages.
3. `run_clustering=False` loads an existing canonical CSV from disk — this is the normal pattern after the first full run.
4. Market data failures skip Phases 6–7 gracefully without crashing the pipeline. The canonical table is always returned.
5. All phase functions are top-level imports in `pipeline.py` — this is what makes them mockable via `patch("signal_intelligence.pipeline.embed_ptcs")`.
6. `PipelineResult` holds DataFrames, not file paths — post-processing notebooks work directly with the return value.

---

## Course complete

You've now covered the full Signal Intelligence pipeline:

| Lesson | Module | Key concept |
|--------|--------|-------------|
| 01 | `ptc_extraction.models` | PTC schema, source_date vs event_date, atomicity |
| 02 | `ptc_extraction.chunking` | Deterministic chunking, offset invariant |
| 03 | `ptc_extraction.extractor` | LLM extraction, validation, resume-safe JSONL |
| 04 | `signal_intelligence.embedding` + `.clustering` | Titan embeddings, HDBSCAN, cross-map |
| 05 | `signal_intelligence.timeseries` | Quarter alignment, mention matrix |
| 06 | `signal_intelligence.correlation` + `.eventstudy` | ADF/BH/Granger, CAR, t-test |
| 07 | `signal_intelligence.pipeline` | End-to-end orchestration, phase toggles |

**Production run:** Set up `.env` with `FMP_API_KEY` and ensure AWS credentials for `eu-west-3` are configured, then `run_pipeline(PipelineConfig(ticker="ML.PA", company_name="Michelin", sector="Specialty Chemicals & Tires", pool_path=Path("data/processed/ptc_pool.jsonl")))`.